<a href="https://colab.research.google.com/github/TinaKgn/tourism_data_project/blob/absa-llm-validation/notebooks/users/kristina/exploratory/02_ABSA_LLM_Sentiment_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ABSA Step 2: LLM-Based Aspect Sentiment Evaluation

**⚠️ Colab-Only Notebook** - This notebook is designed exclusively for Google Colab with GPU acceleration (T4 recommended).

**Purpose**: Evaluate sentiment for detected aspects using LLM (Mistral-7B 4-bit) with hallucination mitigation.

**Input**: Aspect presence results from `01_ABSA_NLI_Aspect_Detection.ipynb`

**Output**: Parquet file with nested dict structure: `{aspect_name: {"present": bool, "sentiment": str, "confidence": float}, ...}`

**Note on confidence**: The `confidence` field is a placeholder (hard-coded 0.8 for deterministic outputs, 0.7 for sampled outputs, 0.0 for not-present). It is not a calibrated probability.

**Workflow**: Validation → Full (configurable via RUN_MODE parameter)

## Configuration

**Modify these parameters to control the pipeline:**

| Parameter | Recommended | Description |
|-----------|-------------|-------------|
| `LLM_APPROACH` | `"transformers"` | Use "transformers" (most reliable on Colab) or "ollama" (alternative) |
| `MODEL_NAME` | Auto-set based on approach | **Transformers**: `"mistralai/Mistral-7B-Instruct-v0.3"`<br>**Ollama**: `"mistral:7b"`<br>⚠️ Update manually when switching approaches |
| `USE_4BIT` | `True` | Enable 4-bit quantization (Transformers only, ~4GB VRAM vs ~14GB) |
| `RUN_MODE` | `"validation"` | Test on 100 reviews first, then switch to `"full"` for all 5000 |
| `TEMPERATURE` | `0.0` | Deterministic outputs (reduces hallucinations) |

In [1]:
# =============================================================================
# IMPORTS & WARNING SUPPRESSION
# =============================================================================

# Suppress unnecessary warnings BEFORE importing libraries
import os
import warnings

# Suppress transformers warnings before importing
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Suppress Python warnings
warnings.filterwarnings('ignore', message='.*pad_token_id.*')
warnings.filterwarnings('ignore', message='.*generation flags.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Now import all dependencies
import json
import hashlib
import pickle
import requests
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from typing import Optional, Dict, List
from pathlib import Path

print("✅ Dependencies loaded with optimized warning suppression")

# =============================================================================
# ⚙️  USER CONFIGURATION - MODIFY AS NEEDED
# =============================================================================

# Choose LLM approach: "ollama" or "transformers"
# Recommended: "transformers" (more reliable on Colab)
LLM_APPROACH = "transformers"  # Options: "ollama", "transformers"

# Model name - UPDATE THIS WHEN CHANGING LLM_APPROACH
# For transformers: use HuggingFace model path (e.g., "mistralai/Mistral-7B-Instruct-v0.3")
# For ollama: use Ollama model tag (e.g., "mistral:7b")
if LLM_APPROACH == "transformers":
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # Transformers format
elif LLM_APPROACH == "ollama":
    MODEL_NAME = "mistral:7b"  # Ollama format
else:
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # Default fallback

# Use 4-bit quantization for Transformers (reduces memory usage on T4 GPU)
# Ignored for Ollama (handles quantization automatically)
USE_4BIT = True

# Run mode: "validation" (limited sample for testing) or "full" (all 5000 reviews)
RUN_MODE = "validation"  # Options: "validation", "full"

# Sample size for validation run (ignored if RUN_MODE == "full")
VALIDATION_SAMPLE_SIZE = 100

# Temperature for LLM inference (0.0 = deterministic, higher = more variable)
TEMPERATURE = 0.0

# Batch size for processing
BATCH_SIZE = 8

# Checkpoint and resume configuration
ENABLE_CHECKPOINTS = True  # Set to False to disable checkpointing
CHECKPOINT_BATCH_SIZE = 500  # Save checkpoint every N reviews (prevents data loss)

print(f"✅ Configuration set:")
print(f"   LLM Approach: {LLM_APPROACH}")
print(f"   Model: {MODEL_NAME}")
print(f"   Run Mode: {RUN_MODE}")
if RUN_MODE == "validation":
    print(f"   Validation Sample Size: {VALIDATION_SAMPLE_SIZE}")
if LLM_APPROACH == "transformers":
    print(f"   4-bit Quantization: {USE_4BIT}")
print(f"   Temperature: {TEMPERATURE}")
if ENABLE_CHECKPOINTS:
    print(f"   Checkpoint Interval: Every {CHECKPOINT_BATCH_SIZE} reviews")
print(f"   Checkpointing: {ENABLE_CHECKPOINTS}")


✅ Dependencies loaded with optimized warning suppression
✅ Configuration set:
   LLM Approach: transformers
   Model: mistralai/Mistral-7B-Instruct-v0.3
   Run Mode: validation
   Validation Sample Size: 100
   4-bit Quantization: True
   Temperature: 0.0
   Checkpoint Interval: Every 500 reviews
   Checkpointing: True


## Setup and Dependencies

In [ ]:
import os
import sys
import pickle
import hashlib
import json
import re
from typing import List, Dict, Optional, Tuple
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests

# =============================================================================
# COLAB SETUP: Mount Google Drive
# =============================================================================

print("🌐 Running on Google Colab")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_BASE = '/content/drive/MyDrive/tourism_data_project'
if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive mount unavailable at /content/drive/MyDrive')
if not os.path.isdir(DRIVE_BASE):
    raise FileNotFoundError(f'Expected project folder not found in Drive: {DRIVE_BASE}')
print(f"✅ Drive path verified: {DRIVE_BASE}")

print("✅ Environment setup complete")

🌐 Running on Google Colab
Mounted at /content/drive
✅ Environment setup complete


In [3]:
# =============================================================================
# Install bitsandbytes for 4-bit quantization (Transformers only)
# =============================================================================
if LLM_APPROACH == "transformers" and USE_4BIT:
    print("📦 Installing bitsandbytes for 4-bit quantization...")
    !pip install -q bitsandbytes accelerate
    print("✅ Bitsandbytes installed successfully")
else:
    print("⏭️  Skipping bitsandbytes install (not needed for current configuration)")

📦 Installing bitsandbytes for 4-bit quantization...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
✅ Bitsandbytes installed successfully


In [4]:
# =============================================================================
# COLAB ONLY: Optional Ollama install + server startup
# =============================================================================
if LLM_APPROACH == "ollama":
    print("⚠️  Colab Ollama setup is best-effort and may be unstable.")
    print(f"    Installing and starting Ollama with model: {MODEL_NAME}")

    # Install Ollama
    !curl -fsSL https://ollama.com/install.sh | sh

    # Start Ollama server in the background
    import subprocess
    import time
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

    # Pull the model specified in configuration
    !ollama pull {MODEL_NAME}
    print(f"✅ Ollama setup complete with model: {MODEL_NAME}")
else:
    print(f"⏭️  Skipping Ollama setup (using {LLM_APPROACH} backend)")

⏭️  Skipping Ollama setup (using transformers backend)


## Load NLI Results from Step 1

In [5]:
# Load NLI results from Step 1
nli_path = os.path.join(DRIVE_BASE, "data/silver/tripadvisor/staged_primary/tripadvisor_nyc_absa_nli_aspects_5000_staged.parquet")
print(f"📂 Loading NLI results from: {nli_path}")

df_nli = pd.read_parquet(nli_path)
print(f"✅ Loaded {len(df_nli)} reviews with NLI aspect detection results")

# Helper function to identify detected aspects
def get_detected_aspects(aspects_dict):
    """Extract list of aspects that were detected (present=True)."""
    if not isinstance(aspects_dict, dict):
        return []
    return [aspect for aspect, details in aspects_dict.items()
            if isinstance(details, dict) and details.get('present', False)]

# Count detected aspects per review
df_nli['detected_aspect_count'] = df_nli['nli_aspects'].apply(
    lambda x: len(get_detected_aspects(x))
)

print(f"\n📊 NLI Results Summary:")
print(f"  Total reviews: {len(df_nli)}")
print(f"  Reviews with ≥1 detected aspect: {(df_nli['detected_aspect_count'] > 0).sum()}")
print(f"  Average aspects per review: {df_nli['detected_aspect_count'].mean():.2f}")

📂 Loading NLI results from: /content/drive/MyDrive/tourism_data_project/data/silver/tripadvisor/staged_primary/tripadvisor_nyc_absa_nli_aspects_5000_staged.parquet
✅ Loaded 5000 reviews with NLI aspect detection results

📊 NLI Results Summary:
  Total reviews: 5000
  Reviews with ≥1 detected aspect: 4997
  Average aspects per review: 6.86


## Cache Directory Setup

In [6]:
# CACHE DIRECTORY SETUP (Google Drive)

cache_dir = os.path.join(DRIVE_BASE, "data/reference/cache/llm_sentiment")
os.makedirs(cache_dir, exist_ok=True)
print(f"✅ Cache directory ready: {cache_dir}")

✅ Cache directory ready: /content/drive/MyDrive/tourism_data_project/data/reference/cache/llm_sentiment


## LLM Sentiment Evaluation Class

In [ ]:
class LLMSentimentEvaluator:
    """
    Evaluates sentiment for detected aspects using LLM.

    Features:
    - Two LLM backends: Ollama and Transformers
    - Disk caching to avoid recomputation
    - Hallucination mitigation (temperature control + explicit guardrails)
    - Batch processing with progress tracking
    - 4-bit quantization support for Transformers (reduces VRAM usage)
    """

    def __init__(
        self,
        llm_approach: str = "ollama",
        model_name: str = None,
        temperature: float = 0.0,
        cache_dir: str = "cache",
        device: Optional[str] = None,
        use_4bit: bool = False
    ):
        """
        Initialize LLM sentiment evaluator.

        Args:
            llm_approach: "ollama" or "transformers"
            model_name: Model identifier (HuggingFace path or Ollama tag)
            temperature: LLM sampling temperature (0.0 = deterministic)
            cache_dir: Directory for caching LLM inference results
            device: Torch device (auto-detected if None)
            use_4bit: Enable 4-bit quantization (Transformers only)
        """
        if llm_approach not in ["ollama", "transformers"]:
            raise ValueError(f"llm_approach must be 'ollama' or 'transformers', got {llm_approach}")

        self.llm_approach = llm_approach
        self.model_name = model_name
        self.temperature = temperature
        self.cache_dir = cache_dir
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_4bit = use_4bit

        os.makedirs(cache_dir, exist_ok=True)

        self.model = None
        self.tokenizer = None

        self._initialize_llm()

    def _initialize_llm(self):
        """Initialize the appropriate LLM backend."""
        if self.llm_approach == "ollama":
            print(f"🦙 Using Ollama backend with model: {self.model_name}")
            print("   Make sure Ollama is running on localhost:11434")
            print(f"   Run setup cell or: ollama pull {self.model_name}")
            # Validate connection
            try:
                response = requests.get("http://localhost:11434/api/tags", timeout=5)
                if response.status_code == 200:
                    try:
                        tags = response.json().get("models", [])
                        model_names = [m.get("name", "") for m in tags if isinstance(m, dict)]
                        # Check if model exists (exact match or prefix match)
                        model_found = any(
                            name == self.model_name or name.startswith(self.model_name.split(':')[0])
                            for name in model_names
                        )
                        if model_found:
                            print("✅ Ollama connection successful & model found")
                        else:
                            print(f"⚠️  Ollama is running, but {self.model_name} is missing")
                            print(f"   Run: ollama pull {self.model_name}")
                    except ValueError:
                        print("⚠️  Ollama connection successful, but could not parse tags")
                else:
                    print(f"⚠️  Ollama returned status {response.status_code}")
            except requests.exceptions.ConnectionError:
                print("❌ Cannot connect to Ollama. Make sure it's running on localhost:11434")
                print("   On Colab, run the Ollama setup cell or use Transformers approach")

        elif self.llm_approach == "transformers":
            print(f"🤗 Using Transformers backend with model: {self.model_name}")
            if self.use_4bit:
                print("   Loading in 4-bit mode (reduced memory footprint)")
            try:
                from transformers import AutoTokenizer, AutoModelForCausalLM

                print(f"   Loading {self.model_name}...")

                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

                # Load model with optional 4-bit quantization
                if self.use_4bit:
                    from transformers import BitsAndBytesConfig
                    quantization_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_compute_dtype=torch.float16,
                        bnb_4bit_use_double_quant=True,
                        bnb_4bit_quant_type="nf4"
                    )
                    self.model = AutoModelForCausalLM.from_pretrained(
                        self.model_name,
                        quantization_config=quantization_config,
                        device_map="auto"
                    )
                else:
                    self.model = AutoModelForCausalLM.from_pretrained(
                        self.model_name,
                        torch_dtype=torch.float16,
                        device_map="auto"
                    )

                self.model.eval()
                print(f"✅ Model loaded successfully on {self.device}")
            except Exception as e:
                print(f"❌ Error loading model: {e}")

    def _hash_item(self, review_idx: int, aspect: str) -> str:
        """Create deterministic hash for cache."""
        key = f"{review_idx}::{aspect}"
        return hashlib.md5(key.encode("utf-8")).hexdigest()

    def _load_cache(self, path: str):
        """Load cached result from disk."""
        if os.path.exists(path):
            try:
                with open(path, "rb") as f:
                    return pickle.load(f)
            except (pickle.PickleError, EOFError):
                os.remove(path)
        return None

    def _save_cache(self, path: str, obj):
        """Save result to disk cache."""
        tmp_path = f"{path}.tmp"
        with open(tmp_path, "wb") as f:
            pickle.dump(obj, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
        if not os.path.exists(path) or os.path.getsize(path) == 0:
            raise IOError(f"Cache write verification failed: {path}")

    def _build_evaluation_prompt(self, review_text: str, aspect: str) -> str:
        """
        Build prompt for aspect sentiment evaluation with hallucination guards.
        """
        prompt = f"""You are an expert evaluator of aspect-based sentiment in tourism reviews.

TASK: Evaluate sentiment for ONE SPECIFIC ASPECT in the provided review.

REVIEW:
"{review_text}"

ASPECT TO EVALUATE: {aspect}

INSTRUCTIONS:
1. Determine if this aspect is CLEARLY DISCUSSED in the review.
2. If discussed, classify the sentiment as: positive, negative, or neutral.
3. IMPORTANT: Only assign a sentiment if you find clear evidence in the review.
4. If the aspect is NOT discussed or the sentiment is UNCLEAR, respond ONLY with: "NOT_PRESENT"

RESPONSE FORMAT:
Respond with ONLY ONE of these:
- positive
- negative
- neutral
- NOT_PRESENT

Do NOT include any explanation. Do NOT include any additional text."""

        return prompt

    def _parse_llm_response(self, response: str) -> Dict[str, any]:
        """
        Parse LLM response into structured format.

        Returns:
            {"sentiment": str or None, "confidence": float, "valid": bool}
        """
        response_clean = response.strip().lower()

        valid_sentiments = ["positive", "negative", "neutral"]
        not_present_keywords = ["not_present", "not present", "unclear", "ambiguous"]
        none_only_tokens = ["none", "null", "n/a", "na"]

        # Check for NOT_PRESENT / none tokens
        if response_clean in none_only_tokens or any(kw in response_clean for kw in not_present_keywords):
            return {"sentiment": None, "confidence": 0.0, "valid": True}

        # Check for valid sentiments
        for sentiment in valid_sentiments:
            if sentiment in response_clean:
                # Extract confidence if response includes it
                confidence = 0.8 if self.temperature == 0.0 else 0.7
                return {"sentiment": sentiment, "confidence": confidence, "valid": True}

        # Invalid response format
        return {"sentiment": None, "confidence": 0.0, "valid": False}

    def evaluate_aspect_sentiment(
        self,
        review_idx: int,
        review_text: str,
        aspect: str
    ) -> Dict[str, any]:
        """
        Evaluate sentiment for a single aspect in a review.

        Returns:
            {"sentiment": str or None, "confidence": float, "cached": bool}
        """
        # Check cache first
        cache_key = self._hash_item(review_idx, aspect)
        cache_path = os.path.join(self.cache_dir, f"{cache_key}.pkl")

        cached_result = self._load_cache(cache_path)
        if cached_result is not None:
            result = cached_result.copy()
            result["cached"] = True
            return result

        # Generate LLM response
        prompt = self._build_evaluation_prompt(review_text, aspect)

        try:
            if self.llm_approach == "ollama":
                response = self._call_ollama(prompt)
            elif self.llm_approach == "transformers":
                response = self._call_transformers(prompt)
            else:
                raise ValueError(f"Unknown LLM approach: {self.llm_approach}")

            # Parse response
            parsed = self._parse_llm_response(response)

            # Save to cache
            self._save_cache(cache_path, {
                "sentiment": parsed["sentiment"],
                "confidence": parsed["confidence"]
            })

            parsed["cached"] = False
            return parsed

        except Exception as e:
            print(f"Error evaluating aspect {aspect} for review {review_idx}: {e}")
            return {"sentiment": None, "confidence": 0.0, "cached": False, "error": str(e)}

    def _call_ollama(self, prompt: str) -> str:
        """
        Call Ollama API for LLM inference.
        """
        url = "http://localhost:11434/api/generate"

        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "temperature": self.temperature,
            "top_p": 0.9,
            "stream": False
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()

        result = response.json()
        return result.get("response", "")

    def _call_transformers(self, prompt: str) -> str:
        """
        Call Transformers-based LLM for inference.
        """
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("Model not initialized")

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            # Suppress warnings during generation
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=50,
                    temperature=max(0.1, self.temperature),  # Avoid temperature=0 in generate
                    top_p=0.9,
                    do_sample=self.temperature > 0,
                    pad_token_id=self.tokenizer.eos_token_id  # Prevent pad_token_id warning
                )

        response_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the generated part (not the prompt)
        response_text = response_text[len(prompt):].strip()

        return response_text

## Prepare Validation or Full Dataset

In [8]:
# Select reviews based on RUN_MODE
if RUN_MODE == "validation":
    # Filter to reviews with at least one detected aspect
    df_with_aspects = df_nli[df_nli['detected_aspect_count'] > 0].copy()
    # Take first N
    df_process = df_with_aspects.head(VALIDATION_SAMPLE_SIZE).reset_index(drop=True)
    print(f"📋 VALIDATION MODE")
    print(f"   Processing {len(df_process)} reviews (first {VALIDATION_SAMPLE_SIZE} with detected aspects)")
elif RUN_MODE == "full":
    df_process = df_nli.copy()
    print(f"📋 FULL MODE")
    print(f"   Processing all {len(df_process)} reviews")
else:
    raise ValueError(f"RUN_MODE must be 'validation' or 'full', got {RUN_MODE}")

print(f"\n   Temperature: {TEMPERATURE}")
print(f"   LLM Approach: {LLM_APPROACH}")

📋 VALIDATION MODE
   Processing 100 reviews (first 100 with detected aspects)

   Temperature: 0.0
   LLM Approach: transformers


## Initialize LLM Evaluator

In [ ]:
import json
from pathlib import Path

def _atomic_write_json(path: str, payload: dict):
    tmp_path = f"{path}.tmp"
    with open(tmp_path, 'w') as f:
        json.dump(payload, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise IOError(f"Checkpoint JSON write verification failed: {path}")

def _atomic_write_pickle(path: str, payload):
    tmp_path = f"{path}.tmp"
    with open(tmp_path, 'wb') as f:
        pickle.dump(payload, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise IOError(f"Checkpoint pickle write verification failed: {path}")

# Initialize evaluator
evaluator = LLMSentimentEvaluator(
    llm_approach=LLM_APPROACH,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    cache_dir=cache_dir,
    use_4bit=USE_4BIT
)

print(f"✅ LLM Evaluator initialized")
print(f"   Model: {MODEL_NAME}")
print(f"   Cache directory: {cache_dir}")

# =============================================================================
# CHECKPOINT SYSTEM INITIALIZATION
# =============================================================================

# Setup checkpoint directory
if ENABLE_CHECKPOINTS:
    checkpoint_dir = os.path.join(DRIVE_BASE, "data/reference/cache/checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Create checkpoint metadata file path
    checkpoint_meta_path = os.path.join(checkpoint_dir, f"metadata_{RUN_MODE}.json")
    checkpoint_results_path = os.path.join(checkpoint_dir, f"results_{RUN_MODE}.pkl")

    print(f"\n💾 Checkpoint system enabled")
    print(f"   Checkpoint directory: {checkpoint_dir}")
    print(f"   Checkpoint interval: Every {CHECKPOINT_BATCH_SIZE} reviews")
    print(f"   Checkpoint metadata path: {checkpoint_meta_path}")
    print(f"   Checkpoint results path: {checkpoint_results_path}")

    # Check if resuming from existing checkpoint
    meta_exists = os.path.exists(checkpoint_meta_path)
    results_exists = os.path.exists(checkpoint_results_path)
    if meta_exists != results_exists:
        print('⚠️  Partial checkpoint detected. Clearing inconsistent files and starting fresh.')
        if meta_exists:
            os.remove(checkpoint_meta_path)
        if results_exists:
            os.remove(checkpoint_results_path)
        meta_exists = False

    if meta_exists:
        with open(checkpoint_meta_path, 'r') as f:
            checkpoint_meta = json.load(f)

        resume_from_idx = checkpoint_meta.get('last_completed_idx', -1)
        with open(checkpoint_results_path, 'rb') as f:
            all_results = pickle.load(f)
        cache_hits = checkpoint_meta.get('cache_hits', 0)
        cache_misses = checkpoint_meta.get('cache_misses', 0)
        error_count = checkpoint_meta.get('error_count', 0)

        print(f"   ✅ RESUMING from checkpoint")
        print(f"      Completed indices: 0-{resume_from_idx}")
        print(f"      Results loaded: {len(all_results)} reviews")
        print(f"      Stats - Hits: {cache_hits}, Misses: {cache_misses}, Errors: {error_count}")
        start_idx = resume_from_idx + 1
    else:
        resume_from_idx = -1
        all_results = []
        cache_hits = 0
        cache_misses = 0
        error_count = 0
        start_idx = 0
        print(f"   ℹ️  Starting fresh (no checkpoint found)")
else:
    print(f"\n⏭️  Checkpointing disabled")
    all_results = []
    cache_hits = 0
    cache_misses = 0
    error_count = 0
    start_idx = 0

## Run Sentiment Evaluation

In [10]:
# =============================================================================
# CHECKPOINT MANAGEMENT UTILITIES (Optional - run for checkpoint info)
# =============================================================================

def check_checkpoint_status():
    """Display checkpoint status and progress"""
    if not ENABLE_CHECKPOINTS:
        print("⏭️  Checkpointing is disabled")
        return

    checkpoint_dir = os.path.join(DRIVE_BASE, "data/reference/cache/checkpoints")
    checkpoint_meta_path = os.path.join(checkpoint_dir, f"metadata_{RUN_MODE}.json")

    if not os.path.exists(checkpoint_meta_path):
        print("No checkpoint found. Notebook will start from scratch.")
        return

    with open(checkpoint_meta_path, 'r') as f:
        meta = json.load(f)

    print("📊 Checkpoint Status:")
    print(f"   Mode: {RUN_MODE}")
    print(f"   Last completed: {meta.get('last_completed_idx', 'N/A')}")
    print(f"   Total reviews: {meta.get('total_reviews', 'N/A')}")
    print(f"   Progress: {(meta.get('last_completed_idx', 0) + 1) / meta.get('total_reviews', 1) * 100:.1f}%")
    print(f"   Cache stats - Hits: {meta.get('cache_hits', 0)}, Misses: {meta.get('cache_misses', 0)}")
    print(f"   Errors: {meta.get('error_count', 0)}")
    print(f"   Last updated: {meta.get('timestamp', 'N/A')}")
    print(f"   Status: {meta.get('status', 'IN_PROGRESS')}")

def clear_checkpoint():
    """Clear existing checkpoint to restart from scratch"""
    if not ENABLE_CHECKPOINTS:
        print("⏭️  Checkpointing is disabled")
        return

    checkpoint_dir = os.path.join(DRIVE_BASE, "data/reference/cache/checkpoints")
    checkpoint_meta_path = os.path.join(checkpoint_dir, f"metadata_{RUN_MODE}.json")
    checkpoint_results_path = os.path.join(checkpoint_dir, f"results_{RUN_MODE}.pkl")

    if os.path.exists(checkpoint_meta_path):
        os.remove(checkpoint_meta_path)
        print(f"🗑️  Deleted: {checkpoint_meta_path}")

    if os.path.exists(checkpoint_results_path):
        os.remove(checkpoint_results_path)
        print(f"🗑️  Deleted: {checkpoint_results_path}")

    if not os.path.exists(checkpoint_meta_path) and not os.path.exists(checkpoint_results_path):
        print("✅ Checkpoint cleared. Notebook will start from scratch on next run.")

# Display current checkpoint status
print("📋 Checkpoint Status Check:")
check_checkpoint_status()
print("\n💡 Utilities available:")
print("   check_checkpoint_status() - Display progress")
print("   clear_checkpoint() - Restart from scratch")

📋 Checkpoint Status Check:
No checkpoint found. Notebook will start from scratch.

💡 Utilities available:
   check_checkpoint_status() - Display progress
   clear_checkpoint() - Restart from scratch


In [ ]:
# Process reviews and evaluate aspect sentiments with batched checkpointing
print(f"\n🚀 Starting sentiment evaluation...")
print(f"   Total reviews to process: {len(df_process) - start_idx}")
if ENABLE_CHECKPOINTS:
    print(f"   Resuming from review index: {start_idx}")
print()

# Iterate through reviews with batched checkpoint saving
total_reviews = len(df_process)

with tqdm(total=total_reviews - start_idx, desc="Processing reviews") as pbar:
    # Update progress bar for already-completed reviews
    if start_idx > 0:
        pbar.update(0)

    for idx in range(start_idx, total_reviews):
        row = df_process.iloc[idx]
        orig_idx = row['orig_index']
        review_text = row['rvw_text_pr']
        nli_aspects = row['nli_aspects']

        # Get detected aspects
        detected_aspects = get_detected_aspects(nli_aspects)

        if len(detected_aspects) == 0:
            # No aspects detected for this review
            all_results.append({
                "orig_index": orig_idx,
                "review_text": review_text,
                "aspects": {}
            })
        else:
            # Initialize result for this review
            review_result = {
                "orig_index": orig_idx,
                "review_text": review_text,
                "aspects": {}
            }

            # Evaluate sentiment for each detected aspect
            for aspect in detected_aspects:
                sentiment_result = evaluator.evaluate_aspect_sentiment(
                    review_idx=orig_idx,
                    review_text=review_text,
                    aspect=aspect
                )

                if not isinstance(sentiment_result, dict):
                    sentiment_result = {
                        "sentiment": None,
                        "confidence": 0.0,
                        "cached": False,
                        "error": "invalid_result"
                    }

                # Normalize sentiment value
                sentiment_value = sentiment_result.get("sentiment")
                if isinstance(sentiment_value, str):
                    cleaned = sentiment_value.strip().lower()
                    if cleaned in {"none", "null", "n/a", "na", "not_present", "not present"}:
                        sentiment_value = None
                    else:
                        sentiment_value = cleaned

                # Normalize confidence to float
                confidence_value = sentiment_result.get("confidence", 0.0)
                try:
                    confidence_value = float(confidence_value)
                except (TypeError, ValueError):
                    confidence_value = 0.0

                # Track cache performance
                if sentiment_result.get("cached", False):
                    cache_hits += 1
                else:
                    cache_misses += 1

                if sentiment_result.get("error"):
                    error_count += 1

                # Store result with presence=True from NLI stage
                review_result["aspects"][aspect] = {
                    "present": True,  # NLI already confirmed presence
                    "sentiment": sentiment_value,
                    "confidence": confidence_value
                }

            all_results.append(review_result)

        pbar.update(1)

        # Save checkpoint at regular intervals
        if ENABLE_CHECKPOINTS and (idx + 1) % CHECKPOINT_BATCH_SIZE == 0:
            checkpoint_meta = {
                "last_completed_idx": idx,
                "total_reviews": total_reviews,
                "cache_hits": cache_hits,
                "cache_misses": cache_misses,
                "error_count": error_count,
                "timestamp": pd.Timestamp.now().isoformat()
            }

            _atomic_write_json(checkpoint_meta_path, checkpoint_meta)
            _atomic_write_pickle(checkpoint_results_path, all_results)

            cache_ratio = f"{cache_hits}/{cache_hits + cache_misses}" if (cache_hits + cache_misses) > 0 else "N/A"
            pbar.write(f"💾 Checkpoint at {idx + 1}/{total_reviews} | Cache: {cache_ratio} | Errors: {error_count}")

# Save final checkpoint after completion
if ENABLE_CHECKPOINTS:
    checkpoint_meta = {
        "last_completed_idx": total_reviews - 1,
        "total_reviews": total_reviews,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "error_count": error_count,
        "timestamp": pd.Timestamp.now().isoformat(),
        "status": "COMPLETE"
    }

    _atomic_write_json(checkpoint_meta_path, checkpoint_meta)
    _atomic_write_pickle(checkpoint_results_path, all_results)

print(f"\n✅ Sentiment evaluation complete")
print(f"   Processed: {len(all_results)} reviews")
print(f"   Cache hits: {cache_hits}")
print(f"   Cache misses: {cache_misses}")
if error_count > 0:
    print(f"   Errors: {error_count}")


🚀 Starting sentiment evaluation...
   Total reviews to process: 100
   Resuming from review index: 0



Processing reviews: 100%|██████████| 100/100 [20:16<00:00, 12.17s/it]


✅ Sentiment evaluation complete
   Processed: 100 reviews
   Cache hits: 0
   Cache misses: 816


In [14]:
# ADD THIS DIAGNOSTIC CELL AFTER "Run Sentiment Evaluation" cell
# Run this to inspect the all_results structure BEFORE saving

print("🔍 DIAGNOSTIC: Inspecting all_results structure before save\n")

# Check first 5 reviews with aspects
sample_count = 0
for idx, result in enumerate(all_results[:20]):
    if not result.get("aspects"):
        continue

    sample_count += 1
    print(f"\nReview {idx} (orig_index={result['orig_index']}):")
    print(f"  Type of aspects: {type(result['aspects'])}")
    print(f"  Number of aspects: {len(result['aspects'])}")

    for aspect_name, aspect_data in result['aspects'].items():
        print(f"\n  Aspect: {aspect_name}")
        print(f"    Type: {type(aspect_data)}")
        print(f"    Value: {aspect_data}")
        print(f"    Is dict: {isinstance(aspect_data, dict)}")

        if isinstance(aspect_data, dict):
            for key, val in aspect_data.items():
                print(f"      {key}: {val} (type: {type(val).__name__})")
        else:
            print(f"    ❌ NOT A DICT! This is the problem!")

    if sample_count >= 3:
        break

print("\n" + "="*80)
print("SUMMARY OF ISSUES FOUND:")
print("="*80)

# Count malformed entries
malformed_count = 0
for result in all_results:
    if result.get("aspects"):
        for aspect_name, aspect_data in result['aspects'].items():
            if not isinstance(aspect_data, dict):
                malformed_count += 1

print(f"\nTotal malformed aspect entries: {malformed_count}")
print(f"Total results: {len(all_results)}")
print(f"Malformed percentage: {malformed_count / sum(len(r.get('aspects', {})) for r in all_results) * 100:.2f}%")

🔍 DIAGNOSTIC: Inspecting all_results structure before save


Review 0 (orig_index=67961):
  Type of aspects: <class 'dict'>
  Number of aspects: 9

  Aspect: accommodation quality
    Type: <class 'dict'>
    Value: {'present': True, 'sentiment': 'negative', 'confidence': 0.8}
    Is dict: True
      present: True (type: bool)
      sentiment: negative (type: str)
      confidence: 0.8 (type: float)

  Aspect: atmosphere
    Type: <class 'dict'>
    Value: {'present': True, 'sentiment': 'positive', 'confidence': 0.8}
    Is dict: True
      present: True (type: bool)
      sentiment: positive (type: str)
      confidence: 0.8 (type: float)

  Aspect: availability
    Type: <class 'dict'>
    Value: {'present': True, 'sentiment': None, 'confidence': 0.0}
    Is dict: True
      present: True (type: bool)
      sentiment: None (type: NoneType)
      confidence: 0.0 (type: float)

  Aspect: cleanliness
    Type: <class 'dict'>
    Value: {'present': True, 'sentiment': 'negative', 'confide

## Merge and Save Results

In [22]:
# REPLACE the "Merge and Save Results" cell with:

import json

# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Define primary schema columns
primary_cols = [
    'rvw_id_pr', 'usr_id_pr', 'lst_id_pr', 'rvw_text_pr', 'rvw_text_flags_pr',
    'rvw_vader_pr', 'rvw_year_pr', 'rvw_month_pr', 'lst_lat_pr', 'lst_lon_pr', 'lst_metro_code_pr',
    'is_accommodation_pr', 'is_restaurant_pr', 'is_nightlife_pr', 'is_entertainment_pr', 'is_tours_pr', 'is_events_pr'
]

# Start with process_df and ensure all primary columns exist
staged = df_process.copy()
for col in primary_cols:
    if col not in staged.columns:
        staged[col] = None

# Convert nested dicts to JSON strings to preserve structure during parquet serialization
staged['llm_aspects'] = results_df['aspects'].apply(lambda x: json.dumps(x) if isinstance(x, dict) else '{}')

# Assemble output columns
output_cols = ['orig_index'] + primary_cols + ['llm_aspects']
staged_output = staged.reindex(columns=output_cols)

# SAVE OUTPUT (Google Drive)
out_dir = os.path.join(DRIVE_BASE, 'data', 'silver', 'tripadvisor', 'staged_primary')
os.makedirs(out_dir, exist_ok=True)

# Filename based on RUN_MODE
if RUN_MODE == "validation":
    out_filename = f'tripadvisor_nyc_absa_llm_validation_n{len(df_process)}_staged.parquet'
else:
    out_filename = 'tripadvisor_nyc_absa_llm_full_5000_staged.parquet'

out_path = os.path.join(out_dir, out_filename)
staged_output.to_parquet(out_path, index=False)
print(f'✅ Saved LLM sentiment results to Drive: {out_path}')

✅ Saved LLM sentiment results to Drive: /content/drive/MyDrive/tourism_data_project/data/silver/tripadvisor/staged_primary/tripadvisor_nyc_absa_llm_validation_n100_staged.parquet


## Verify Output

In [23]:
# REPLACE the "Verify Output" cell with:

# Load and verify the saved results
df_llm = pd.read_parquet(out_path)

# Deserialize JSON strings back to dicts
df_llm['llm_aspects'] = df_llm['llm_aspects'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})

print(f"✅ Loaded LLM results: {len(df_llm)} rows")
print(f"Columns: {df_llm.columns.tolist()}\n")

# Sample output inspection
print("Sample LLM aspects structure (first review with aspects):")
for idx, aspects_dict in enumerate(df_llm['llm_aspects']):
    if isinstance(aspects_dict, dict) and len(aspects_dict) > 0:
        print(f"\nReview {idx}:")
        for aspect, details in list(aspects_dict.items())[:3]:  # Show first 3 aspects
            print(f"  {aspect}: {details}")
        break

# Count sentiments
def extract_sentiments(aspects_dict):
    if not isinstance(aspects_dict, dict):
        return []
    return [v.get('sentiment') for v in aspects_dict.values() if isinstance(v, dict)]

df_llm['aspect_sentiments'] = df_llm['llm_aspects'].apply(extract_sentiments)

print(f"\n📊 LLM Sentiment Evaluation Summary:")
print(f"  Total reviews processed: {len(df_llm)}")
print(f"  Reviews with ≥1 sentiment: {(df_llm['aspect_sentiments'].apply(len) > 0).sum()}")

# Sentiment distribution
all_sentiments = []
for sentiments_list in df_llm['aspect_sentiments']:
    all_sentiments.extend([s for s in sentiments_list if s is not None])

if all_sentiments:
    sentiment_counts = pd.Series(all_sentiments).value_counts()
    print(f"\n  Sentiment Distribution:")
    for sentiment, count in sentiment_counts.items():
        pct = 100 * count / len(all_sentiments)
        print(f"    {sentiment}: {count} ({pct:.1f}%)")
else:
    print(f"\n  ⚠️  No sentiments extracted. Check LLM output format and parser.")

✅ Loaded LLM results: 100 rows
Columns: ['orig_index', 'rvw_id_pr', 'usr_id_pr', 'lst_id_pr', 'rvw_text_pr', 'rvw_text_flags_pr', 'rvw_vader_pr', 'rvw_year_pr', 'rvw_month_pr', 'lst_lat_pr', 'lst_lon_pr', 'lst_metro_code_pr', 'is_accommodation_pr', 'is_restaurant_pr', 'is_nightlife_pr', 'is_entertainment_pr', 'is_tours_pr', 'is_events_pr', 'llm_aspects']

Sample LLM aspects structure (first review with aspects):

Review 0:
  accommodation quality: {'present': True, 'sentiment': 'negative', 'confidence': 0.8}
  atmosphere: {'present': True, 'sentiment': 'positive', 'confidence': 0.8}
  availability: {'present': True, 'sentiment': None, 'confidence': 0.0}

📊 LLM Sentiment Evaluation Summary:
  Total reviews processed: 100
  Reviews with ≥1 sentiment: 100

  Sentiment Distribution:
    positive: 606 (78.1%)
    neutral: 115 (14.8%)
    negative: 55 (7.1%)


In [25]:
print("🔍 DIAGNOSTIC: Comparing all_results vs loaded parquet\n")

# Load the saved parquet
df_llm = pd.read_parquet(out_path)

# ⚠️ CRITICAL: Deserialize JSON strings back to dicts immediately after loading
df_llm['llm_aspects'] = df_llm['llm_aspects'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else {}
)

print("="*80)
print("BEFORE SAVE (all_results in memory):")
print("="*80)

sample_review = all_results[0]
print(f"\nReview 0 llm_aspects structure:")
print(f"  Type: {type(sample_review['aspects'])}")
print(f"  Keys: {list(sample_review['aspects'].keys())[:3]}")

sample_aspect_name = list(sample_review['aspects'].keys())[0]
sample_aspect_data = sample_review['aspects'][sample_aspect_name]
print(f"\nSample aspect '{sample_aspect_name}':")
print(f"  Type: {type(sample_aspect_data)}")
print(f"  Value: {sample_aspect_data}")

print("\n" + "="*80)
print("AFTER LOAD (from parquet + DESERIALIZATION):")
print("="*80)

loaded_aspects = df_llm.iloc[0]['llm_aspects']
print(f"\nReview 0 llm_aspects structure:")
print(f"  Type: {type(loaded_aspects)}")

if isinstance(loaded_aspects, dict):
    print(f"  Keys: {list(loaded_aspects.keys())[:3]}")

    sample_loaded_aspect_name = list(loaded_aspects.keys())[0]
    sample_loaded_aspect_data = loaded_aspects[sample_loaded_aspect_name]
    print(f"\nSample aspect '{sample_loaded_aspect_name}':")
    print(f"  Type: {type(sample_loaded_aspect_data)}")
    print(f"  Value: {sample_loaded_aspect_data}")

    if isinstance(sample_loaded_aspect_data, dict):
        for key, val in sample_loaded_aspect_data.items():
            print(f"    {key}: {val} (type: {type(val).__name__})")
else:
    print(f"  ❌ llm_aspects is still a string! Deserialization failed.")

print("\n" + "="*80)
print("CHECKING FOR MALFORMED ENTRIES IN LOADED DATA:")
print("="*80)

malformed_entries = []
for idx, row in df_llm.iterrows():
    llm_aspects = row['llm_aspects']
    if not isinstance(llm_aspects, dict):
        malformed_entries.append({
            'row_idx': idx,
            'type': type(llm_aspects),
            'value': str(llm_aspects)[:100]  # Truncate for display
        })
        continue

    for aspect_name, aspect_data in llm_aspects.items():
        if not isinstance(aspect_data, dict):
            malformed_entries.append({
                'row_idx': idx,
                'aspect_name': aspect_name,
                'type': type(aspect_data),
                'value': aspect_data
            })

if malformed_entries:
    print(f"\n❌ Found {len(malformed_entries)} malformed entries:")
    for entry in malformed_entries[:10]:
        print(f"  {entry}")
else:
    print(f"\n✅ No malformed entries found in loaded data!")

🔍 DIAGNOSTIC: Comparing all_results vs loaded parquet

BEFORE SAVE (all_results in memory):

Review 0 llm_aspects structure:
  Type: <class 'dict'>
  Keys: ['accommodation quality', 'atmosphere', 'availability']

Sample aspect 'accommodation quality':
  Type: <class 'dict'>
  Value: {'present': True, 'sentiment': 'negative', 'confidence': 0.8}

AFTER LOAD (from parquet + DESERIALIZATION):

Review 0 llm_aspects structure:
  Type: <class 'dict'>
  Keys: ['accommodation quality', 'atmosphere', 'availability']

Sample aspect 'accommodation quality':
  Type: <class 'dict'>
  Value: {'present': True, 'sentiment': 'negative', 'confidence': 0.8}
    present: True (type: bool)
    sentiment: negative (type: str)
    confidence: 0.8 (type: float)

CHECKING FOR MALFORMED ENTRIES IN LOADED DATA:

✅ No malformed entries found in loaded data!


## Next Steps

## Checkpoint & Resume System

### How It Works
The notebook now includes **checkpoint/resume logic** that automatically saves progress every `CHECKPOINT_BATCH_SIZE` reviews (default: 500). This allows you to:
- **Resume interrupted runs** without losing progress if Colab disconnects
- **Process large datasets** across multiple sessions
- **Monitor progress** with checkpoint timestamp and statistics

### Configuration
| Parameter | Default | Notes |
|-----------|---------|-------|
| `ENABLE_CHECKPOINTS` | `True` | Disable to skip checkpointing |
| `CHECKPOINT_BATCH_SIZE` | `500` | Save progress every N reviews |
| Checkpoint location | `data/reference/cache/checkpoints/` | In Google Drive (persists across sessions) |

### Workflow: Full Mode (5000 Reviews)

#### Session 1: Process first batch
1. Run all cells with `RUN_MODE = "full"`
2. Notebook saves checkpoint every 500 reviews
3. Set `TEMPERATURE = 0.0` for deterministic outputs
4. If interrupted: Nothing is lost (latest checkpoint saved)

#### Session 2 (if needed): Resume from checkpoint
1. **Re-run configuration and setup cells** (they detect and load checkpoint automatically)
2. Run the main processing cell again — it will:
   - Detect existing checkpoint
   - Load previously completed results
   - **Resume from last completed review** (not the beginning!)
   - Continue processing and saving new batches
3. Repeat until complete (status in checkpoint file shows "COMPLETE")

### Checkpoint Files
Located in: `data/reference/cache/checkpoints/`
- `metadata_full.json`: Progress metadata (last completed index, cache stats, timestamp)
- `results_full.json`: Pickled results up to checkpoint
- `metadata_validation.json`: Same structure for validation mode

### Clearing & Restarting
To **restart from scratch** (ignore existing checkpoint):
```python
# Before running main processing cell, delete:
import shutil
checkpoint_dir = os.path.join(DRIVE_BASE, "data/reference/cache/checkpoints")
shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir)
```

---

## Next Steps

### If you ran in VALIDATION mode:
1. Review the sample output and check a few reviews manually to validate sentiment accuracy
2. If results look good, re-run this notebook with `RUN_MODE = "full"` to process all 5000 reviews
3. If results need adjustment (e.g., too many hallucinations), adjust `TEMPERATURE` or modify the `_build_evaluation_prompt()` method

### If you ran in FULL mode:
1. Results are saved to Google Drive and ready for downstream analysis
2. Load combined results with both NLI presence and LLM sentiment for correlation analysis
3. Checkpoint system preserves progress—if interrupted, simply run cells again to resume

### Model Switching:
- **Default (Recommended)**: Transformers with Mistral-7B 4-bit is most reliable on Colab T4
- **If Transformers fails**: Switch to `LLM_APPROACH = "ollama"` and set `MODEL_NAME = "mistral:7b"` (may require re-running Ollama setup cell)
- **For larger models**: Disable 4-bit by setting `USE_4BIT = False` (may cause OOM on T4)

### Cache Note:
- All LLM inference results are cached in your Google Drive (`data/reference/cache/llm_sentiment`)
- If your Colab runtime disconnects, cached results will be preserved (no re-computation needed)
- To clear cache and re-run evaluations: delete the cache folder from Drive